In [36]:
%pip install langchain-tavily

Note: you may need to restart the kernel to use updated packages.


In [37]:
import os
from pathlib import Path
from typing import TypedDict

import requests
from dotenv import load_dotenv

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_groq import ChatGroq
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_tavily import TavilySearch
from langgraph.graph import StateGraph, START, END


In [38]:
load_dotenv()

GROQ_API_KEY = os.getenv("GROQ_API_KEY")
REWRITEAI_API_KEY = os.getenv("REWRITEAI_API_KEY")
api_key = os.getenv("REVERSEGPT_API_KEY")
TAVILY_API_KEY = os.getenv("TAVILY_API_KEY")
web_domains = [
    domain.strip()
    for domain in os.getenv(
        "WEB_SEARCH_DOMAIN", "wikipedia.org"
    ).split(",")
    if domain.strip()
]

In [39]:
BASE_DIR =Path.cwd()
DATA_DIR = BASE_DIR / "data"
DATA_DIR.mkdir(exist_ok=True)

In [40]:
llm = ChatGroq(
    model="openai/gpt-oss-120b",
    api_key=os.environ["GROQ_API_KEY"],
    temperature=1,
    reasoning_effort="medium"
)

search_tool = TavilySearch(
        max_results=5,
        include_domains=web_domains,
        search_depth="basic"
)

In [41]:
class EssayState(TypedDict, total=False):
    question: str
    web_results: str
    original_essay: str
    humanized_essay: str
    detector_score: float
    detector_verdict: str
    detector_confidence: float

In [43]:
def web_search_node(state: EssayState):
    print("[Node] web_search_node")

    results = search_tool.invoke({
        "query": state["question"]
    })

    return {
        "web_results": str(results)
    }

In [44]:
def write_essay_node(state: EssayState):
    print("[Node] write_essay_node")

    prompt = f"""
    {state["question"]}

    Use the web search results below to write the essay:
    {state["web_results"]}

    Instructions:
    - Follow the requested word count.
    - Include a suitable title.
    - Keep the essay accurate and relevant.
    - Return only the essay.
    """

    response = llm.invoke([
        SystemMessage(content="You are a research-based essay writer."),
        HumanMessage(content=prompt),
    ])

    return {"original_essay": response.content}

In [45]:
import os
import time
import requests
from dotenv import load_dotenv
load_dotenv()


def humanize_node(state: EssayState):
    print("[Node] humanize_node")


    base_url = "https://www.reversegpt.ai/api/v1"
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json",
    }

    # Start the humanization job
    response = requests.post(
        f"{base_url}/humanize",
        headers=headers,
        json={"text": state["original_essay"]},
        timeout=30,
    )
    response.raise_for_status()

    data = response.json()
    job = data["job"]

    # Poll until the rewrite is completed
    deadline = time.monotonic() + 300

    while job["status"] == "pending":
        if time.monotonic() >= deadline:
            raise TimeoutError("ReverseGPT humanization timed out.")

        time.sleep(2)

        job_url = job.get("url")
        if not job_url:
            raise ValueError(f"Missing job URL: {data}")

        poll_response = requests.get(
            job_url,
            headers=headers,
            timeout=30,
        )
        poll_response.raise_for_status()

        data = poll_response.json()
        job = data["job"]

    if job["status"] == "failed":
        raise RuntimeError(
            data.get("error", {}).get(
                "message", "ReverseGPT humanization failed."
            )
        )

    if job["status"] != "completed":
        raise RuntimeError(f"Unexpected ReverseGPT response: {data}")

    humanized_text = data["result"]["humanizedText"]

    print("Original essay characters:", len(state["original_essay"]))
    print("Humanized essay characters:", len(humanized_text))

    return {
        "humanized_essay": humanized_text
    }

In [50]:
def ai_checker_node(state: EssayState):
    """Check the humanized essay using WasItAIGenerated."""
    print("[Node] ai_checker_node")

    url = (
        "https://www.wasitaigenerated.com"
        "/api/v1/detect/text"
    )

    headers = {
        "Authorization": f"Bearer {WASITAI_API_KEY}",
        "Content-Type": "application/json",
    }

    payload = {
        "content": state["humanized_essay"]
    }

    response = requests.post(
        url,
        headers=headers,
        json=payload,
        timeout=120,
    )

    response.raise_for_status()
    data = response.json()

    if "score" not in data:
        raise ValueError(
            f"Detector response has no score: {data}"
        )

    return {
        "detector_score": float(data["score"]),
        "detector_verdict": str(data.get("verdict", "unknown")),
        "detector_confidence": float(data.get("confidence", 0.0)),
    }


In [51]:
builder = StateGraph(EssayState)

builder.add_node("web_search", web_search_node)
builder.add_node("write_essay", write_essay_node)
builder.add_node("humanize", humanize_node)
builder.add_node("ai_checker", ai_checker_node)

builder.add_edge(START, "web_search")
builder.add_edge("web_search", "write_essay")
builder.add_edge("write_essay", "humanize")
builder.add_edge("humanize", "ai_checker")
builder.add_edge("ai_checker", END)

graph = builder.compile()

In [55]:

question = "Write me an essay about dolphins in about 50 words."
result = graph.invoke({
    "question": question
})

original = result.get("original_essay", "")
humanized = result.get("humanized_essay", "")

print("\n" + "=" * 60)
print("ORIGINAL AI-GENERATED ESSAY")
print("=" * 60)
print(original)

print("\n" + "=" * 60)
print("HUMANIZED ESSAY")
print("=" * 60)
print(humanized)

print("\n" + "=" * 60)
print("TEXT COMPARISON")
print("=" * 60)
print("Original characters:", len(original))
print("Humanized characters:", len(humanized))
print("Text changed:", original.strip() != humanized.strip())

print("\n" + "=" * 60)
print("AI DETECTOR RESULT")
print("=" * 60)
print("Raw score:", result.get("detector_score"))
print("Verdict:", result.get("detector_verdict"))
print("Confidence:", result.get("detector_confidence"))

print("\nNote: Detector results are estimates, not proof of authorship.")


[Node] web_search_node
[Node] write_essay_node
[Node] humanize_node
Original essay characters: 456
Humanized essay characters: 410
[Node] ai_checker_node

ORIGINAL AI-GENERATED ESSAY
**Dolphins: The Intelligent Oceanic Mammals**

Dolphins, members of the infraorder Cetacea, include roughly 40 extant species across families such as Delphinidae, Platanistidae, Iniidae and Pontoporiidae. Renowned for social intelligence, complex communication, and playful behavior, they inhabit seas and rivers worldwide, inspiring countless films and cultural depictions. Their echolocation aids hunting, and conservation efforts strive to protect them.

HUMANIZED ESSAY
Dolphins are marine mammals that belong to the infrasuborder Cetacea, there are about 40 species which include Delphinidae, Platanistidae, Iniidae and Pontoporiidae. Dolphins are smart animals, friendly and highly social, they live in oceans all over the world. They're famous for their intelligence and they have been the subject of a number 

NameError: name 'text_to_check' is not defined